## Experiments with different models

**Import libraries**

In [4]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import mean_squared_error, r2_score
import joblib
from sklearn.metrics import precision_recall_curve, auc
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.optimizers import Adam, SGD
from keras.models import load_model

In [6]:
def prepare_data(data):
   # data['size'] = data['size'].astype(str)
   # data_dummies = pd.get_dummies(data, columns=['size'])
    # X = data_dummies.drop(['sample', 'regulator', 'target', 'interaction', 'size'], axis=1).values
    # y = data_dummies['interaction'].values
    X = data.drop(['sample', 'regulator', 'target', 'interaction', 'size'], axis=1).values
    y = data['interaction'].values
    X = np.asarray(X).astype(np.float32)
    y = np.asarray(y).astype(np.float32)
    return X, y

In [ ]:
def performance_metrics(y, y_predicted):
    tp=fp=tn=fn=0
    for i in range(len(y)):
        if y[i]==y_predicted[i]:
            if y[i]==1:
                tp+=1
            else:
                tn+=1
        else:
            if y[i]==1:
                #fp+=1
                fn+=1
            else:
                #fn+=1
                fp+=1
    precision = tp/(tp+fp) if (tp+fp)!=0 else 0
    recall = tp/(tp+fn) if (tp+fn)!=0 else 0
    structural = (tp+tn)/(tp+fp+fn+tn)
    tpr = recall
    fpr = fp/(fp+tn) if (fp+tn)!=0 else 1
    if precision==0 and recall==0:
        f1 = 0
    else:
        f1 = 2*precision*recall/(precision+recall)
    return precision, recall, structural, tpr, fpr, f1

In [ ]:
def get_performace(_test_set, y_predicted):
    test_set = _test_set.copy()
    test_set['predicted'] = y_predicted
    result={}
    sample = set(test_set['sample'].values)
    for sp in sample:
        network = test_set[test_set['sample']==sp]
        p,r,st, tpr, fpr, f1 = performance_metrics(network['interaction'].values, network['predicted'].values)
        result[f'{sp}'] = {'precision':round(p, 4), 'recall':round(r,4), 'structural': round(st,4), 
                           'tpr':round(tpr, 4), 'fpr':round(fpr, 4), 'f1':round(f1, 4)}
    return result

In [ ]:
def get_round(y_predicted, threshold=0.3):
    rounded_predicted = np.where(y_predicted >= threshold, np.ceil(y_predicted), np.floor(y_predicted))
    
    return rounded_predicted

In [ ]:
def get_roc_curve(y, y_predicted):
    tp=fp=tn=fn=0
    for i in range(len(y)):
        if y[i]==y_predicted[i]:
            if y[i]==1:
                tp+=1
            else:
                tn+=1
        else:
            if y[i]==1:
                fp+=1
            else:
                fn+=1
    TPR = tp/(tp+fn)
    FPR = fp/(fp+tn)
    return TPR, FPR

**Data preparation**

In [6]:
def load_size10():
    training_set = pd.read_csv(r'./caocao/training/size10.csv', index_col=0)
    
    training_set_50_1 = pd.read_csv(r'./caocao/training/size50_10.csv', index_col=0)
    training_set_50_2 = pd.read_csv(r'./caocao/training/size50_20.csv', index_col=0)
    
    training_set_50_1_true = training_set_50_1[training_set_50_1['interaction']==1]
    training_set_50_2_true = training_set_50_2[training_set_50_2['interaction']==1]
    
    training_set_sum = pd.concat([training_set, training_set_50_1_true, training_set_50_2_true], ignore_index=True)
    valid_set = pd.read_csv(r'./caocao/validation/size10.csv', index_col=0)
    X_train, y_train = prepare_data(training_set_sum)
    X_valid, y_valid = prepare_data(valid_set)
    return X_train, y_train, X_valid, y_valid

In [6]:
def load_size50():
    '''
    Dataset contains all size10, size50 and true positive of size 100
    '''
    training_set_10 = pd.read_csv(r'./caocao/training/size10.csv', index_col=0)
    training_set_50_1 = pd.read_csv(r'./caocao/training/size50_10.csv', index_col=0)
    training_set_50_2 = pd.read_csv(r'./caocao/training/size50_20.csv', index_col=0)

    training_set_100_1 = pd.read_csv(r'./caocao/training/size100_5.csv', index_col=0)
    training_set_100_2 = pd.read_csv(r'./caocao/training/size100_10.csv', index_col=0)
    training_set_100_3 = pd.read_csv(r'./caocao/training/size100_15.csv', index_col=0)
    training_set_100_4 = pd.read_csv(r'./caocao/training/size100_20.csv', index_col=0)
    
    training_set_100_1_true = training_set_100_1[training_set_100_1['interaction']==1]
    training_set_100_2_true = training_set_100_2[training_set_100_2['interaction']==1]
    training_set_100_3_true = training_set_100_3[training_set_100_3['interaction']==1]
    training_set_100_4_true = training_set_100_4[training_set_100_4['interaction']==1]

    
    training_set_sum = pd.concat([training_set_10, training_set_50_1, training_set_50_2,
                                 training_set_100_1_true, training_set_100_2_true,
                                 training_set_100_3_true, training_set_100_4_true], ignore_index=True)
    valid_set = pd.read_csv(r'./caocao/validation/size50.csv', index_col=0)
    training_set_sum = training_set_sum.dropna()
    X_train, y_train = prepare_data(training_set_sum)
    X_valid, y_valid = prepare_data(valid_set)
    return X_train, y_train, X_valid, y_valid

In [7]:
def load_size100():
    '''
    Dataset contains all true size10, size50 and all size 100
    '''
    training_set_10 = pd.read_csv(r'./caocao/training/size10.csv', index_col=0)
    training_set_10_true = training_set_10[training_set_10['interaction']==1]
    
    training_set_50_1 = pd.read_csv(r'./caocao/training/size50_10.csv', index_col=0)
    training_set_50_2 = pd.read_csv(r'./caocao/training/size50_20.csv', index_col=0)
    training_set_50_1_true = training_set_50_1[training_set_50_1['interaction']==1]
    training_set_50_2_true = training_set_50_2[training_set_50_2['interaction']==1]

    training_set_100_1 = pd.read_csv(r'./caocao/training/size100_5.csv', index_col=0)
    training_set_100_2 = pd.read_csv(r'./caocao/training/size100_10.csv', index_col=0)
    training_set_100_3 = pd.read_csv(r'./caocao/training/size100_15.csv', index_col=0)
    training_set_100_4 = pd.read_csv(r'./caocao/training/size100_20.csv', index_col=0)
    training_set_100_4 = training_set_100_4[training_set_100_4['interaction']==1]
    
    training_set_sum = pd.concat([training_set_10_true, training_set_50_1_true, training_set_50_2_true,
                                 training_set_100_1, training_set_100_2,
                                 training_set_100_3, training_set_100_4], ignore_index=True)
    training_set_sum = training_set_sum.dropna()
    valid_set = pd.read_csv(r'./caocao/validation/size100.csv', index_col=0)
    X_train, y_train = prepare_data(training_set_sum)
    X_valid, y_valid = prepare_data(valid_set)
    return X_train, y_train, X_valid, y_valid

In [8]:
X_train, y_train, X_valid, y_valid = load_size10()

In [9]:
X_train, y_train, X_valid, y_valid = load_size100()

In [11]:
test_set = pd.read_csv(r'./caocao/test/size10.csv', index_col=0)
X_test, y_test = prepare_data(test_set)

In [9]:
def get_report(model, report_filename, size, sample_number_max):
    limit = size*(size-1)
    with open(report_filename, 'w+') as f:
        for i in range(1, sample_number_max+1):
            test_set_report = pd.read_csv(fr'./jump3_code/data for comparison/size{size}/size{size}_{i}.csv', index_col=0)
            X_test_report, y_test_report = prepare_data(test_set_report)
            y_test_report_pred = model.predict(X_test_report)
            precision1, recall1, _ = precision_recall_curve(y_test_report[:limit], y_test_report_pred[:limit])
            aupr1 = auc(recall1, precision1)
            
            precision2, recall2, _ = precision_recall_curve(y_test_report[limit:], y_test_report_pred[limit:])
            aupr2 = auc(recall2, precision2)
            f.write(f'{aupr1}\n')
            f.write(f'{aupr2}\n')

In [18]:
def get_report_random(report_filename, size, sample_number_max):
    limit = size*(size-1)
    with open(report_filename, 'w+') as f:
        for i in range(1, sample_number_max+1):
            test_set_report = pd.read_csv(fr'./jump3_code/data for comparison/size{size}/size{size}_{i}.csv', index_col=0)
            X_test_report, y_test_report = prepare_data(test_set_report)
            aupr1=0
            aupr2=0
            for sp in range(100):
                y_test_report_pred = np.random.rand(limit*2)
                precision1, recall1, _ = precision_recall_curve(y_test_report[:limit], y_test_report_pred[:limit])
                aupr1 += auc(recall1, precision1)
                
                precision2, recall2, _ = precision_recall_curve(y_test_report[limit:], y_test_report_pred[limit:])
                aupr2 += auc(recall2, precision2)
            f.write(f'{aupr1/100}\n')
            f.write(f'{aupr2/100}\n')

In [24]:
get_report_random('./caocao/report_update/model50/random_size50.txt', 50, 4)
get_report_random('./caocao/report_update/model50/random_size10.txt', 10, 10)
get_report_random('./caocao/report_update/model50/random_size100.txt', 100, 4)

### 0. MLP

In [10]:
model = keras.models.Sequential([ 
    keras.layers.Dense(1024, activation="relu", input_shape=X_train.shape[1:]),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(512, activation="relu"),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(256, activation="relu"),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(16, activation="relu"),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(8, activation="relu"),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(1, activation="sigmoid")
])

#optimizer = keras.optimizers.RMSprop(lr=0.001, rho=0.9)
optimizer = SGD(clipvalue=1.0, momentum=0.9, nesterov=True)
#optimizer=Adam(learning_rate=0.01, beta_1=0.9, beta_2=0.999)
early_stopping_cb = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)
model.compile(loss="mean_squared_error", optimizer=optimizer, metrics=['accuracy'])

/home/cao/anaconda3/lib/python3.12/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [11]:
history = model.fit(X_train, y_train, epochs=100, validation_data=(X_valid, y_valid), 
                    callbacks=[early_stopping_cb])

Epoch 1/100
28926/62744 ━━━━━━━━━━━━━━━━━━━━ 5:16 9ms/step - accuracy: 0.7683 - loss: 0.1576

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



57401/62744 ━━━━━━━━━━━━━━━━━━━━ 49s 9ms/step - accuracy: 0.7700 - loss: 0.1563

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



60123/62744 ━━━━━━━━━━━━━━━━━━━━ 24s 9ms/step - accuracy: 0.7742 - loss: 0.1528

...

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



22380/62744 ━━━━━━━━━━━━━━━━━━━━ 6:16 9ms/step - accuracy: 0.7802 - loss: 0.1494

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



29748/62744 ━━━━━━━━━━━━━━━━━━━━ 5:07 9ms/step - accuracy: 0.7812 - loss: 0.1491

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [12]:
model.save('./caocao/trained_model/model_10.h5')

In [16]:
model.save('./caocao/trained_model/model_50.h5')

In [ ]:
model = load_model('model_50_add_10all_100true.h5')

In [ ]:
model = load_model('model_10_add_50true.h5')

In [16]:
get_report(model, './caocao/report_update/model100/mlp_size50.txt', 50, 4)
get_report(model, './caocao/report_update/model100/mlp_size10.txt', 10, 10)
get_report(model, './caocao/report_update/model100/mlp_size100.txt', 100, 4)

154/154 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
154/154 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
154/154 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
154/154 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
619/619 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step
619/619 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step
619/619 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step
619/619 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step


In [ ]:
get_report(model, './caocao/report_update/model10/mlp_size50.txt', 50, 4)
get_report(model, './caocao/report_update/model10/mlp_size10.txt', 10, 10)
get_report(model, './caocao/report_update/model10/mlp_size100.txt', 100, 4)

154/154 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
154/154 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
154/154 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
154/154 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
619/619 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step
541/619 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step

### 1. Linear Regression

In [17]:
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

LinearRegression()

**Save trained model**

In [19]:
joblib.dump(lin_reg, './caocao/trained_model/lg_10.pkl')

['./caocao/trained_model/lg_10.pkl']

**Load saved model**

In [ ]:
lin_reg = joblib.load('linear_regression_model.pkl')

**Evaluate model**

In [ ]:
# Predict on the validation set
y_val_pred = lin_reg.predict(X_valid)

mse_val = mean_squared_error(y_valid, y_val_pred)
r2_val = r2_score(y_valid, y_val_pred)

print(f'Validation MSE: {mse_val}')
print(f'Validation R^2: {r2_val}')

In [ ]:
# Predict on the test set
y_test_pred = lin_reg.predict(X_test)

mse_test = mean_squared_error(y_test, y_test_pred)
r2_test = r2_score(y_test, y_test_pred)

print(f'Test MSE: {mse_test}')
print(f'Test R^2: {r2_test}')

In [ ]:
get_report(lin_reg, './caocao/report/model100/linear_size10.txt', 10, 10)

In [ ]:
get_report(lin_reg, './caocao/report/model100/linear_size50.txt', 50, 4)

In [ ]:
get_report(lin_reg, './caocao/report/model100/linear_size100.txt', 100, 4)

In [30]:
get_report(lin_reg, './caocao/report_update/model50/lr_size50.txt', 50, 4)
get_report(lin_reg, './caocao/report_update/model50/lr_size10.txt', 10, 10)
get_report(lin_reg, './caocao/report_update/model50/lr_size100.txt', 100, 4)

In [24]:
get_report(lin_reg, './caocao/report_update/model100/lr_size50.txt', 50, 4)
get_report(lin_reg, './caocao/report_update/model100/lr_size10.txt', 10, 10)
get_report(lin_reg, './caocao/report_update/model100/lr_size100.txt', 100, 4)

In [21]:
get_report(lin_reg, './caocao/report_update/model10/lr_size50.txt', 50, 4)
get_report(lin_reg, './caocao/report_update/model10/lr_size10.txt', 10, 10)
get_report(lin_reg, './caocao/report_update/model10/lr_size100.txt', 100, 4)

**ROC curve**

In [ ]:
thresholds = [i/100 for i in range(100, 0, -1)]
performances = []
for threshold in thresholds:
    predicted = get_round(y_test_pred, threshold)
    result = get_performace(test_set, predicted)
    performance = pd.DataFrame.from_dict(result, orient='index')
    performances.append(performance)
    performance.to_csv(fr'./caocao/others/Linear/size10_threshold_{threshold}.csv')

In [ ]:
curves_info = {i:[[],[]] for i in range(100)}
for per in performances:
    for i in range(100):
        curves_info[i][0].append(per.iloc[i, 3])
        curves_info[i][1].append(per.iloc[i, 4])

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 6))
plt.rcParams.update({
    'font.size': 12,              # Default font size for text
    'axes.titlesize': 20,         # Font size for axes titles
    'axes.labelsize': 15,         # Font size for x and y labels
    'xtick.labelsize': 12,        # Font size for x tick labels
    'ytick.labelsize': 12,        # Font size for y tick labels
    'legend.fontsize': 12,        # Font size for legend
    'figure.titlesize': 22        # Font size for figure title
})
plt.tick_params(axis='both',        # Apply changes to both x and y axes
                which='major',      # Apply changes to major ticks
                direction='inout',    # Ticks pointing outwards
                length=10,          # Length of ticks
                width=2           # Width of ticks
                #colors='red',       # Color of ticks
                #grid_color='black', # Color of gridlines
                #grid_alpha=0.5)     # Transparency of gridlines
               )
plt.xlim(0, 0.35)
plt.ylim(0, 0.8)
for k, v in curves_info.items():
    
    plt.plot(v[1], v[0], linewidth=3, marker='o', linestyle='dashed', color='#ff4f00')   
    plt.fill_between(v[1], v[0], color='#99ce3e', alpha=0.4)
    plt.text(0.2, 0.3, 'AUC', fontsize=14, color='#0d929a', ha='center', weight='bold')
    plt.text(0.07, 0.6, 'ROC', fontsize=14, color='#0d929a', ha='center',  weight='bold')
    plt.plot([0,v[1][-1]], [0, v[0][-1]], linewidth=3, linestyle='solid', color='#650000')
    plt.title(f'ROC curve - sample {k}')
    plt.xlabel('FP Rate')
    plt.ylabel('TP Rate')
    plt.savefig(fr'./caocao/others/Linear/ROC/s10_{k}.png')
    plt.clf()

### 2. SVM

In [79]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
svm_model = SVC(kernel='rbf', random_state=42)

# Train the model
svm_model.fit(X_train, y_train)

**Save trained model**

In [ ]:
joblib.dump(svm_model, 'svm_model_size50_add_10all_100true.pkl')

In [ ]:
joblib.dump(svm_model, './caocao/trained_model/svm_10.pkl')

**Load saved model**

In [ ]:
svm_model = joblib.load('svm_model.pkl')

**Evaluate model**

In [ ]:
# Predict on the validation set
y_val_pred = svm_model.predict(X_valid)

accuracy = accuracy_score(y_valid, y_val_pred)
print(f'Validation Accuracy: {accuracy}')

# More detailed performance metrics
print(classification_report(y_valid, y_val_pred))
print(confusion_matrix(y_valid, y_val_pred))

In [ ]:
# Predict on the test set
y_test_pred = svm_model.predict(X_test)

accuracy = accuracy_score(y_test, y_test_pred)
print(f'Test Accuracy: {accuracy}')

# More detailed performance metrics
print(classification_report(y_test, y_test_pred))
print(confusion_matrix(y_test, y_test_pred))

In [ ]:
get_report(svm_model, './caocao/report/svm_size10.txt', 10, 10)
get_report(svm_model, './caocao/report/svm_size50.txt', 50, 4)
get_report(svm_model, './caocao/report/svm_size100.txt', 100, 4)

In [ ]:
get_report(svm_model, './caocao/report_update/model10/svm_size50.txt', 50, 4)
get_report(svm_model, './caocao/report_update/model10/svm_size10.txt', 10, 10)
get_report(svm_model, './caocao/report_update/model10/svm_size100.txt', 100, 4)

### 3. Decision Tree

In [23]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [25]:
dt_model = DecisionTreeClassifier(random_state=42)

# Train the model
dt_model.fit(X_train, y_train)

DecisionTreeClassifier(random_state=42)

**Save trained model**

In [ ]:
joblib.dump(dt_model, 'decision_tree_model_size100_add_10true_50true.pkl')

In [26]:
joblib.dump(dt_model, './caocao/trained_model/dt_10.pkl')

['./caocao/trained_model/dt_10.pkl']

**Load saved model**

In [ ]:
# Load the model from the file
dt_model = joblib.load('decision_tree_model.pkl')

In [ ]:
get_report(dt_model, './caocao/report/model100/dt_size50.txt', 50, 4)
get_report(dt_model, './caocao/report/model100/dt_size10.txt', 10, 10)
get_report(dt_model, './caocao/report/model100/dt_size100.txt', 100, 4)

In [42]:
get_report(dt_model, './caocao/report_update/model50/dt_size50.txt', 50, 4)
get_report(dt_model, './caocao/report_update/model50/dt_size10.txt', 10, 10)
get_report(dt_model, './caocao/report_update/model50/dt_size100.txt', 100, 4)

In [30]:
get_report(dt_model, './caocao/report_update/model100/dt_size50.txt', 50, 4)
get_report(dt_model, './caocao/report_update/model100/dt_size10.txt', 10, 10)
get_report(dt_model, './caocao/report_update/model100/dt_size100.txt', 100, 4)

In [27]:
get_report(dt_model, './caocao/report_update/model10/dt_size50.txt', 50, 4)
get_report(dt_model, './caocao/report_update/model10/dt_size10.txt', 10, 10)
get_report(dt_model, './caocao/report_update/model10/dt_size100.txt', 100, 4)

**Evaluate model**

In [ ]:
# Predict on the validation set
y_val_pred = dt_model.predict(X_valid)

# Evaluate the model's performance
accuracy = accuracy_score(y_valid, y_val_pred)
print(f'Validation Accuracy: {accuracy}')

# More detailed performance metrics
print(classification_report(y_valid, y_val_pred))
print(confusion_matrix(y_valid, y_val_pred))

In [ ]:
get_report(dt_model, './caocao/report/dtree_size10.txt', 10, 10)
get_report(dt_model, './caocao/report/dtree_size50.txt', 50, 4)
get_report(dt_model, './caocao/report/dtree_size100.txt', 100, 4)

### 4. Random Forest

In [31]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [33]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# Train the model
rf_model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

**Save trained model**

In [ ]:
joblib.dump(rf_model, 'random_forest_model_size100_add_10all_50true.pkl')

In [34]:
joblib.dump(rf_model, './caocao/trained_model/rf_10.pkl')

['./caocao/trained_model/rf_10.pkl']

**Load saved model**

In [ ]:
rf_model = joblib.load('random_forest_model.pkl')

**Evaluate model**

In [ ]:
get_report(rf_model, './caocao/report/model100/rf_size50.txt', 50, 4)
get_report(rf_model, './caocao/report/model100/rf_size10.txt', 10, 10)
get_report(rf_model, './caocao/report/model100/rf_size100.txt', 100, 4)

In [48]:
get_report(rf_model, './caocao/report_update/model50/rf_size50.txt', 50, 4)
get_report(rf_model, './caocao/report_update/model50/rf_size10.txt', 10, 10)
get_report(rf_model, './caocao/report_update/model50/rf_size100.txt', 100, 4)

In [38]:
get_report(rf_model, './caocao/report_update/model100/rf_size50.txt', 50, 4)
get_report(rf_model, './caocao/report_update/model100/rf_size10.txt', 10, 10)
get_report(rf_model, './caocao/report_update/model100/rf_size100.txt', 100, 4)

In [35]:
get_report(rf_model, './caocao/report_update/model10/rf_size50.txt', 50, 4)
get_report(rf_model, './caocao/report_update/model10/rf_size10.txt', 10, 10)
get_report(rf_model, './caocao/report_update/model10/rf_size100.txt', 100, 4)

In [ ]:
# Predict on the validation set
y_val_pred = rf_model.predict(X_valid)

# Evaluate the model's performance
accuracy = accuracy_score(y_valid, y_val_pred)
print(f'Validation Accuracy: {accuracy}')

# More detailed performance metrics
print(classification_report(y_valid, y_val_pred))
print(confusion_matrix(y_valid, y_val_pred))

In [ ]:
get_report(rf_model, './caocao/report/rforest_size10.txt', 10, 10)
get_report(rf_model, './caocao/report/rforest_size50.txt', 50, 4)
get_report(rf_model, './caocao/report/rforest_size100.txt', 100, 4)

### 5. K-Nearest Neighbors

In [39]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

**Feature Scaling (Important for KNN)**

In [41]:
knn_model = KNeighborsClassifier(n_neighbors=10)
# k=10 is the best
# Train the model
knn_model.fit(X_train, y_train)

KNeighborsClassifier(n_neighbors=10)

**Save trained model**

In [ ]:
joblib.dump(knn_model, 'knn_model_size100_add_10all_50true.pkl')

In [43]:
joblib.dump(knn_model, './caocao/trained_model/knn_10.pkl')

['./caocao/trained_model/knn_10.pkl']

**Load saved model**

In [ ]:
knn_model = joblib.load('knn_model.pkl')

**Evaluate model**

In [ ]:
# Predict on the validation set
y_val_pred = knn_model.predict(X_valid)

accuracy = accuracy_score(y_valid, y_val_pred)
print(f'Validation Accuracy: {accuracy}')

print(classification_report(y_valid, y_val_pred))
print(confusion_matrix(y_valid, y_val_pred))

In [ ]:
get_report(knn_model, './caocao/report/knn_size10.txt', 10, 10)
get_report(knn_model, './caocao/report/knn_size50.txt', 50, 4)
get_report(knn_model, './caocao/report/knn_size100.txt', 100, 4)

In [ ]:
get_report(knn_model, './caocao/report/model100/knn_size50.txt', 50, 4)
get_report(knn_model, './caocao/report/model100/knn_size10.txt', 10, 10)
get_report(knn_model, './caocao/report/model100/knn_size100.txt', 100, 4)

In [13]:
get_report(knn_model, './caocao/report_update/model50/knn_size50.txt', 50, 4)
get_report(knn_model, './caocao/report_update/model50/knn_size10.txt', 10, 10)
get_report(knn_model, './caocao/report_update/model50/knn_size100.txt', 100, 4)

In [19]:
get_report(knn_model, './caocao/report_update/model100/knn_size50.txt', 50, 4)
get_report(knn_model, './caocao/report_update/model100/knn_size10.txt', 10, 10)
get_report(knn_model, './caocao/report_update/model100/knn_size100.txt', 100, 4)

In [45]:
get_report(knn_model, './caocao/report_update/model10/knn_size50.txt', 50, 4)
get_report(knn_model, './caocao/report_update/model10/knn_size10.txt', 10, 10)
get_report(knn_model, './caocao/report_update/model10/knn_size100.txt', 100, 4)

**Hyperparameter Tuning**

In [ ]:
# Try different values for n_neighbors
for k in range(1, 11):
    knn_model = KNeighborsClassifier(n_neighbors=k)
    knn_model.fit(X_train, y_train)
    y_val_pred = knn_model.predict(X_valid)
    accuracy = accuracy_score(y_valid, y_val_pred)
    print(f'Validation Accuracy with k={k}: {accuracy}')

### 6. XGBoost

In [ ]:
!pip install xgboost

In [47]:
import xgboost as xgb

**Convert data to DMatrix Format**

In [49]:
# Convert the datasets into DMatrix, which is the data structure that XGBoost uses
dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_valid, label=y_valid)

In [50]:
# Set up the parameters
params = {
    'max_depth': 3,         # Maximum depth of a tree
    'eta': 0.1,             # Learning rate
    'objective': 'binary:logistic',  # Binary classification objective
    'eval_metric': 'logloss' # Evaluation metric
}

# Optionally, you can add more parameters like:
# 'subsample': 0.8,  # Subsample ratio of the training instances
# 'colsample_bytree': 0.8,  # Subsample ratio of columns when constructing each tree

In [53]:
# Specify validation set for monitoring performance
evallist = [(dtrain, 'train'), (dval, 'eval')]

# Train the model
num_round = 100  # Number of boosting rounds
bst = xgb.train(params, dtrain, num_round, evals=evallist, early_stopping_rounds=10)

[0]	train-logloss:0.53780	eval-logloss:0.43175
[1]	train-logloss:0.52869	eval-logloss:0.42357
[2]	train-logloss:0.52114	eval-logloss:0.41684
[3]	train-logloss:0.51478	eval-logloss:0.41121
[4]	train-logloss:0.50953	eval-logloss:0.40626
[5]	train-logloss:0.50508	eval-logloss:0.40232
[6]	train-logloss:0.50133	eval-logloss:0.39882
[7]	train-logloss:0.49814	eval-logloss:0.39590
[8]	train-logloss:0.49539	eval-logloss:0.39308
[9]	train-logloss:0.49302	eval-logloss:0.39098
[10]	train-logloss:0.49097	eval-logloss:0.38902
[11]	train-logloss:0.48922	eval-logloss:0.38737
[12]	train-logloss:0.48770	eval-logloss:0.38574
[13]	train-logloss:0.48637	eval-logloss:0.38427
[14]	train-logloss:0.48517	eval-logloss:0.38326
[15]	train-logloss:0.48410	eval-logloss:0.38231
[16]	train-logloss:0.48317	eval-logloss:0.38130
[17]	train-logloss:0.48239	eval-logloss:0.38042
[18]	train-logloss:0.48164	eval-logloss:0.37983
[19]	train-logloss:0.48100	eval-logloss:0.37907
[20]	train-logloss:0.48041	eval-logloss:0.37868
[2

**Save trained model**

In [ ]:
bst.save_model('xgboost_model_size_add_10all_100true.json')

In [54]:
bst.save_model('./caocao/trained_model/xgb_10.json')

**Load saved model**

In [ ]:
loaded_bst = xgb.Booster()
loaded_bst.load_model('xgboost_model.json')

**Evaluate model**

In [ ]:
# Make predictions on the validation set
y_val_pred_prob = bst.predict(dval)
y_val_pred = [1 if prob > 0.5 else 0 for prob in y_val_pred_prob]

# Evaluate the model's performance
accuracy = accuracy_score(y_valid, y_val_pred)
print(f'Validation Accuracy: {accuracy}')

# More detailed performance metrics
print(classification_report(y_valid, y_val_pred))
print(confusion_matrix(y_valid, y_val_pred))

In [57]:
def get_report_xgb(model, report_filename, size, sample_number_max):
    limit = size*(size-1)
    with open(report_filename, 'w+') as f:
        for i in range(1, sample_number_max+1):
            test_set_report = pd.read_csv(fr'./jump3_code/data for comparison/size{size}/size{size}_{i}.csv', index_col=0)
            X_test_report, y_test_report = prepare_data(test_set_report)
            dval = xgb.DMatrix(X_test_report, label=y_test_report)
            y_test_report_pred = model.predict(dval)
            precision1, recall1, _ = precision_recall_curve(y_test_report[:limit], y_test_report_pred[:limit])
            aupr1 = auc(recall1, precision1)
            
            precision2, recall2, _ = precision_recall_curve(y_test_report[limit:], y_test_report_pred[limit:])
            aupr2 = auc(recall2, precision2)
            f.write(f'{aupr1}\n')
            f.write(f'{aupr2}\n')

In [ ]:
#test size10
get_report_xgb(loaded_bst, './caocao/report/xgb_size10.txt', 10, 10)
get_report_xgb(loaded_bst, './caocao/report/xgb_size50.txt', 50, 4)
get_report_xgb(loaded_bst, './caocao/report/xgb_size100.txt', 100, 4)

In [ ]:
#test size 50
get_report_xgb(bst, './caocao/report/model100/xgb_size10.txt', 10, 10)
get_report_xgb(bst, './caocao/report/model100/xgb_size50.txt', 50, 4)
get_report_xgb(bst, './caocao/report/model100/xgb_size100.txt', 100, 4)

In [65]:
get_report_xgb(bst, './caocao/report_update/model50/xgb_size50.txt', 50, 4)
get_report_xgb(bst, './caocao/report_update/model50/xgb_size10.txt', 10, 10)
get_report_xgb(bst, './caocao/report_update/model50/xgb_size100.txt', 100, 4)

In [32]:
get_report_xgb(bst, './caocao/report_update/model100/xgb_size50.txt', 50, 4)
get_report_xgb(bst, './caocao/report_update/model100/xgb_size10.txt', 10, 10)
get_report_xgb(bst, './caocao/report_update/model100/xgb_size100.txt', 100, 4)

In [59]:
get_report_xgb(bst, './caocao/report_update/model10/xgb_size50.txt', 50, 4)
get_report_xgb(bst, './caocao/report_update/model10/xgb_size10.txt', 10, 10)
get_report_xgb(bst, './caocao/report_update/model10/xgb_size100.txt', 100, 4)

### 7. LightGBM

In [ ]:
!pip install lightgbm

In [61]:
import lightgbm as lgb

**Convert data to LightGBM dataset format**

In [63]:
train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_valid, label=y_valid, reference=train_data)

In [65]:
# Set up the parameters
params = {
    'boosting_type': 'gbdt',  # Gradient Boosting Decision Tree
    'objective': 'binary',    # Binary classification
    'metric': 'binary_logloss', # Metric to evaluate
    'num_leaves': 31,         # Maximum tree leaves for base learners
    'learning_rate': 0.05,    # Learning rate
    'feature_fraction': 0.9   # Fraction of features to be used for each tree
}

# You can add more parameters like:
# 'bagging_fraction': 0.8,  # Subsample ratio of the training instances
# 'bagging_freq': 5,        # Frequency for bagging
# 'max_depth': -1,          # Maximum depth of the tree (unlimited if set to -1)

In [67]:
# Train the model
lgb_model = lgb.train(params, train_data, num_boost_round=100,\
                valid_sets=[train_data, val_data])

[LightGBM] [Info] Number of positive: 476100, number of negative: 1531700
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.431496 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 26010
[LightGBM] [Info] Number of data points in the train set: 2007800, number of used features: 102
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.237125 -> initscore=-1.168506
[LightGBM] [Info] Start training from score -1.168506


**Save trained model**

In [68]:
# Save the model to a file
lgb_model.save_model('./caocao/trained_model/lgb_10.json')

**Load saved model**

In [ ]:
# Load the model from a file
loaded_bst = lgb.Booster(model_file='lightgbm_model.txt')

**Evaluate model**

In [ ]:
# Make predictions on the validation set
y_val_pred_prob = bst.predict(X_valid, num_iteration=bst.best_iteration)
y_val_pred = [1 if prob > 0.5 else 0 for prob in y_val_pred_prob]

# Evaluate the model's performance
accuracy = accuracy_score(y_valid, y_val_pred)
print(f'Validation Accuracy: {accuracy}')

# More detailed performance metrics
print(classification_report(y_valid, y_val_pred))
print(confusion_matrix(y_valid, y_val_pred))

In [69]:
def get_report_lgb(model, report_filename, size, sample_number_max):
    limit = size*(size-1)
    with open(report_filename, 'w+') as f:
        for i in range(1, sample_number_max+1):
            test_set_report = pd.read_csv(fr'./jump3_code/data for comparison/size{size}/size{size}_{i}.csv', index_col=0)
            X_test_report, y_test_report = prepare_data(test_set_report)
            y_test_report_pred = model.predict(X_test_report, num_iteration=model.best_iteration)
            precision1, recall1, _ = precision_recall_curve(y_test_report[:limit], y_test_report_pred[:limit])
            aupr1 = auc(recall1, precision1)
            
            precision2, recall2, _ = precision_recall_curve(y_test_report[limit:], y_test_report_pred[limit:])
            aupr2 = auc(recall2, precision2)
            f.write(f'{aupr1}\n')
            f.write(f'{aupr2}\n')

In [ ]:
get_report_lgb(bst, './caocao/report/model100/lgb_size10.txt', 10, 10)
get_report_lgb(bst, './caocao/report/model100/lgb_size50.txt', 50, 4)
get_report_lgb(bst, './caocao/report/model100/lgb_size100.txt', 100, 4)

In [83]:
get_report_lgb(lgb_model, './caocao/report_update/model50/lgb_size50.txt', 50, 4)
get_report_lgb(lgb_model, './caocao/report_update/model50/lgb_size10.txt', 10, 10)
get_report_lgb(lgb_model, './caocao/report_update/model50/lgb_size100.txt', 100, 4)

In [46]:
get_report_lgb(lgb_model, './caocao/report_update/model100/lgb_size50.txt', 50, 4)
get_report_lgb(lgb_model, './caocao/report_update/model100/lgb_size10.txt', 10, 10)
get_report_lgb(lgb_model, './caocao/report_update/model100/lgb_size100.txt', 100, 4)

In [73]:
get_report_lgb(lgb_model, './caocao/report_update/model10/lgb_size50.txt', 50, 4)
get_report_lgb(lgb_model, './caocao/report_update/model10/lgb_size10.txt', 10, 10)
get_report_lgb(lgb_model, './caocao/report_update/model10/lgb_size100.txt', 100, 4)

### 8.Naive Bayes Classifier

In [75]:
nb_classifier = GaussianNB()
nb_classifier.fit(X_train, y_train)

GaussianNB()

In [ ]:
get_report(nb_classifier, './caocao/report/nb_size50.txt', 50, 4)
get_report(nb_classifier, './caocao/report/nb_size10.txt', 10, 10)
get_report(nb_classifier, './caocao/report/nb_size100.txt', 100, 4)

In [ ]:
get_report(nb_classifier, './caocao/report/model50/nb_size50.txt', 50, 4)
get_report(nb_classifier, './caocao/report/model50/nb_size10.txt', 10, 10)
get_report(nb_classifier, './caocao/report/model50/nb_size100.txt', 100, 4)

In [ ]:
get_report(nb_classifier, './caocao/report/model100/nb_size50.txt', 50, 4)
get_report(nb_classifier, './caocao/report/model100/nb_size10.txt', 10, 10)
get_report(nb_classifier, './caocao/report/model100/nb_size100.txt', 100, 4)

In [90]:
get_report(nb_classifier, './caocao/report_update/model50/nb_size50.txt', 50, 4)
get_report(nb_classifier, './caocao/report_update/model50/nb_size10.txt', 10, 10)
get_report(nb_classifier, './caocao/report_update/model50/nb_size100.txt', 100, 4)

In [53]:
get_report(nb_classifier, './caocao/report_update/model100/nb_size50.txt', 50, 4)
get_report(nb_classifier, './caocao/report_update/model100/nb_size10.txt', 10, 10)
get_report(nb_classifier, './caocao/report_update/model100/nb_size100.txt', 100, 4)

In [77]:
get_report(nb_classifier, './caocao/report_update/model10/nb_size50.txt', 50, 4)
get_report(nb_classifier, './caocao/report_update/model10/nb_size10.txt', 10, 10)
get_report(nb_classifier, './caocao/report_update/model10/nb_size100.txt', 100, 4)

### 9. LSTM

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
# Normalize the feature data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.fit_transform(X_val)